# KNN regression (nearest neighbors)

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
To predict a house's price, look at the **k most similar houses** and report the mean of their prices. KNN builds no model; it **memorizes** the training data and searches it when asked to predict (lazy learning).

$$\hat y(x)=\frac1k\sum_{i\in N_k(x)}y_i$$

## Steps
- Compute the distance from the new point to every training point (usually Euclidean, $\sqrt{\sum(x_j-x'_j)^2}$).
- Pick the k closest points.
- Return the mean (or a weighted mean) of their $y$.

## A worked example
Size and price of five houses. Predict the price of a 90 m² house with $k=3$:

| Size | Price | Distance to 90 |
|---|---|---|
| 60 | 5.0 | 30 |
| 80 | 6.5 | **10** |
| 95 | 7.4 | **5** |
| 100 | 7.9 | **10** |
| 150 | 11.0 | 60 |

The three neighbors are 95, 80 and 100. Prediction: $(7.4+6.5+7.9)/3=7.27$.

With **distance weights** (`weights="distance"`), closer neighbors get a larger weight $1/d$: $(7.4/5+6.5/10+7.9/10)/(1/5+1/10+1/10)=7.33$.

## The effect of k
> [Small k: a jumpy step curve (overfitting). Large k: a smooth flat curve (underfitting).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/knn_reg_k.png)

*Small k: a jumpy step curve (overfitting). Large k: a smooth flat curve (underfitting).*

| k | Bias | Variance |
|---|---|---|
| 1 | Low | Very high; follows every bit of noise |
| Medium | Balanced | Balanced |
| Close to n | High; the overall mean everywhere | Low |

[[lab:reg_knn]]



In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor

size = np.array([[60], [80], [95], [100], [150]])
price = np.array([5.0, 6.5, 7.4, 7.9, 11.0])

for w in ["uniform", "distance"]:
    m = KNeighborsRegressor(n_neighbors=3, weights=w).fit(size, price)
    print(w, "->", m.predict([[90]]).round(2))
dist, idx = m.kneighbors([[90]])
print("neighbors:", size[idx].ravel(), " distances:", dist.round(1))



## Important notes
- **Scaling is required:** the distance is decided by the feature with the biggest numbers (see the scaling lesson).
- **Curse of dimensionality:** in high dimensions all points are almost equally far, and KNN weakens.
- **No extrapolation:** outside the range of the data the prediction stays flat.
- Prediction is slow, because each new point needs the distance to all the data.



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())
grid = GridSearchCV(pipe, {"kneighborsregressor__n_neighbors": [1, 3, 5, 10, 20, 40], "kneighborsregressor__weights": ["uniform", "distance"]}, cv=5, scoring="r2")
grid.fit(X, y)
print("Best:", grid.best_params_, " R2 =", round(grid.best_score_, 3))
for k, s in zip(grid.cv_results_["params"], grid.cv_results_["mean_test_score"]):
    print(k["kneighborsregressor__n_neighbors"], k["kneighborsregressor__weights"], round(s, 3))

### Exercise

Implement KNN regression from scratch with numpy and reproduce 7.27.

In [ ]:
# Your answer here